In [94]:
import time
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset,DataLoader
import torch.nn.functional as F
from torchvision import datasets,transforms
from torch.optim.lr_scheduler import ReduceLROnPlateau


In [95]:
device=torch.device("cuda"if torch.cuda.is_available() else "cpu")

In [96]:
transform=transforms.Compose([transforms.Resize((224,224)),transforms.ToTensor()])
train_dataset=datasets.ImageFolder(root=r"C:\Users\nishi\code\DL\COVID-19\train",transform=transform)
train_loader=DataLoader(train_dataset,batch_size=16,shuffle=True)
test_dataset=datasets.ImageFolder(root=r"C:\Users\nishi\code\DL\COVID-19\test",transform=transform)
test_loader=DataLoader(test_dataset,batch_size=16,shuffle=False)

In [97]:
for image, label in train_loader:
    print(f"image:{image.size()}")
    print(f"label:{label.size()}")
    break

image:torch.Size([16, 3, 224, 224])
label:torch.Size([16])


In [98]:
class ResnetBlock(torch.nn.Module):
    def __init__(self):
        super(ResnetBlock,self).__init__()
        self.block=torch.nn.Sequential(
            torch.nn.Conv2d(in_channels=3,out_channels=32,kernel_size=(3,3),padding=1),
            torch.nn.BatchNorm2d(32),
            torch.nn.ReLU(),
            torch.nn.Conv2d(32,64,3,padding=1),
            torch.nn.BatchNorm2d(64),
            
        )
        self.shortcut=torch.nn.Sequential(
            torch.nn.Conv2d(in_channels=3,out_channels=64,kernel_size=(3,3),padding=1),
            torch.nn.BatchNorm2d(64),
        )

    def forward(self,x):
        
        block=self.block(x)
        shortcut=self.shortcut(x)
        x=F.relu(block+shortcut)
        
        return x


In [99]:
class Resnet(torch.nn.Module):
    def __init__(self):
        super(Resnet,self).__init__()
        self.resnet_block=ResnetBlock()
        self.pool=torch.nn.AdaptiveAvgPool2d((1,1))
        self.fc=torch.nn.Linear(64,2)

    def forward(self,x):
        x=self.resnet_block(x)
        x=self.pool(x)
        x=x.view(x.size(0),-1)
        x=self.fc(x)
        
        return x

In [100]:
def eval(model,device,data):
    
    model.eval()
    num_examples,correct=0,0

    for features,targets in data:
        features=features.to(device)
        targets=targets.to(device)

        logits=model(features)
        _,prediction=torch.max(logits,dim=1)
        num_examples+=targets.size(0)
        correct+=(prediction==targets).sum()

    return correct.float()/num_examples * 100
def train(model,device,train_loader,num_epochs,optimizer,scheduler):
    minibatch_loss,train_acc_list,val_acc_list=[],[],[]
    start_time=time.time()
    for epoch in range(num_epochs):
        model.train()

        for batch_idx,(features,targets) in enumerate(train_loader):
            features=features.to(device)
            targets=targets.to(device)

            logits=model(features)
            loss=F.cross_entropy(logits,targets)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            minibatch_loss.append(loss.item())

            if not batch_idx % 4:
                print(f'Epoch: {epoch+1:03d}/{num_epochs} |'
                      f'Batch: {batch_idx:04d}/{len(train_loader):04d} |'
                      f'loss: {loss.item():.4f}')
        
        model.eval()
        with torch.no_grad():
            train_acc=eval(model,device,train_loader)
           
            print(f'Epoch: {epoch+1:03d}/{num_epochs} |'
                  f'Train Accuracy: {train_acc:.2f}% |')
            train_acc_list.append(train_acc)
            scheduler.step(train_acc)
        time_elapsed=time.time()-start_time
        print(f'elapsed time: {time_elapsed:.03f}')
    return minibatch_loss,train_acc_list

In [103]:
model=Resnet().to(device=device)
optimizer=torch.optim.Adam(model.parameters(),lr=0.0001)
scheduler = ReduceLROnPlateau(optimizer, "max", factor=0.5, patience=3)
minibatch_loss,train_acc_list=train(model,device,train_loader,
                                    num_epochs=20,optimizer=optimizer,scheduler=scheduler)

Epoch: 001/20 |Batch: 0000/0010 |loss: 0.7707
Epoch: 001/20 |Batch: 0004/0010 |loss: 0.7383
Epoch: 001/20 |Batch: 0008/0010 |loss: 0.7126
Epoch: 001/20 |Train Accuracy: 40.54% |
elapsed time: 11.978
Epoch: 002/20 |Batch: 0000/0010 |loss: 0.6805
Epoch: 002/20 |Batch: 0004/0010 |loss: 0.6621
Epoch: 002/20 |Batch: 0008/0010 |loss: 0.6686
Epoch: 002/20 |Train Accuracy: 50.00% |
elapsed time: 22.901
Epoch: 003/20 |Batch: 0000/0010 |loss: 0.6690
Epoch: 003/20 |Batch: 0004/0010 |loss: 0.6641
Epoch: 003/20 |Batch: 0008/0010 |loss: 0.6302
Epoch: 003/20 |Train Accuracy: 50.00% |
elapsed time: 34.354
Epoch: 004/20 |Batch: 0000/0010 |loss: 0.6638
Epoch: 004/20 |Batch: 0004/0010 |loss: 0.6221
Epoch: 004/20 |Batch: 0008/0010 |loss: 0.6320
Epoch: 004/20 |Train Accuracy: 53.38% |
elapsed time: 45.517
Epoch: 005/20 |Batch: 0000/0010 |loss: 0.6809
Epoch: 005/20 |Batch: 0004/0010 |loss: 0.6163
Epoch: 005/20 |Batch: 0008/0010 |loss: 0.5928
Epoch: 005/20 |Train Accuracy: 63.51% |
elapsed time: 57.035
Epoch

In [104]:
test_acc=eval(model,device,test_loader)
print(f'Test Accuracy: {test_acc}')

Test Accuracy: 100.0
